In [1]:
import pandas as pd
import json

X = pd.read_parquet("x.reduced.parquet")
y = pd.read_parquet("y.reduced.parquet")

with open("moons_split.json") as f:
    splits = json.load(f)

# Separate splits
X_train = X[X["moon"].isin(splits["train"])]
y_train = y[y["moon"].isin(splits["train"])]

X_local_test = X[X["moon"].isin(splits["reduced_local"])]
y_local_test = y[y["moon"].isin(splits["reduced_local"])]

In [2]:
import joblib
import os
from lightgbm import LGBMRegressor  # or any model

def train(X_train, y_train, model_directory_path, loop_moon, embargo):
    # Merge on moon + id
    df = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"])
    
    # Drop rows with missing target
    df = df.dropna(subset=["target"])
    
    feature_cols = [f"Feature_{i}" for i in range(1, 101)]
    
    X = df[feature_cols].fillna(0)
    y = df["target"]
    
    model = LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
    model.fit(X, y)
    
    joblib.dump(model, os.path.join(model_directory_path, "model.joblib"))

In [3]:
def infer(X_test, model_directory_path, loop_moon, embargo):
    model = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    
    feature_cols = [f"Feature_{i}" for i in range(1, 101)]
    
    preds = model.predict(X_test[feature_cols].fillna(0))
    
    return pd.DataFrame({
        "moon": X_test["moon"].values,
        "id": X_test["id"].values,
        "prediction": preds,
    })

In [4]:
embargo = 4
model_dir = "./model"
os.makedirs(model_dir, exist_ok=True)

# Train once on whatever train data you have
train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

# Simulate infer loop over reduced_local moons
results = []
for moon in splits["reduced_local"]:
    X_moon = X_local_test[X_local_test["moon"] == moon]
    pred = infer(X_moon, model_dir, loop_moon=moon, embargo=embargo)
    results.append(pred)

predictions = pd.concat(results)

# Evaluate with Pearson correlation per moon
from scipy.stats import pearsonr

for moon in splits["reduced_local"]:
    p = predictions[predictions["moon"] == moon]
    gt = y_local_test[y_local_test["moon"] == moon]
    merged = p.merge(gt, on=["moon", "id"])
    if len(merged) > 1:
        corr, _ = pearsonr(merged["prediction"], merged["target"])
        print(f"Moon {moon}: Pearson r = {corr:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.071240 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 700
[LightGBM] [Info] Number of data points in the train set: 234011, number of used features: 100
[LightGBM] [Info] Start training from score 0.000000
Moon 773: Pearson r = 0.1001
Moon 774: Pearson r = -0.0002
Moon 775: Pearson r = 0.0685
Moon 776: Pearson r = 0.0468
Moon 777: Pearson r = 0.0707
Moon 778: Pearson r = 0.0889
Moon 779: Pearson r = 0.0410
Moon 780: Pearson r = 0.0024
Moon 781: Pearson r = 0.0332
